# 병합 데이터 컬럼명 및 순서 정리

## Goal

`merged_data.csv`의 39개 컬럼을 짧고 의미가 분명한 한글명으로 바꾸고, **번호 → 시간 → feature → 오프라인 → 결과** 순으로 정렬한다. 원본 데이터는 수정하지 않는다.

## Setup

노트북을 어느 폴더에서 실행해도 프로젝트 루트를 자동으로 찾는다.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'interim' / 'merged_data.csv').exists():
            return candidate
    raise FileNotFoundError('data/interim/merged_data.csv를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / 'data' / 'interim' / 'merged_data.csv'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'interim' / 'merged_data_ko.csv'
DICTIONARY_PATH = PROJECT_ROOT / 'outputs' / '데이터 전처리' / '컬럼_정의.csv'

merged = pd.read_csv(INPUT_PATH)
print(f'입력: {INPUT_PATH}')
print(f'크기: {merged.shape[0]:,}행 × {merged.shape[1]}열')

## Steps

### 1. 컬럼명을 한글로 변경

단위는 컬럼명에 남기고, 이진 플래그는 값의 의미를 함께 표시한다.

In [ ]:
COLUMN_MAP = {
    'Batch_ID': '배치번호',
    'Batch ref': '배치참조번호',
    'Time (h)': '발효시간(h)',
    '0 - Recipe driven 1 - Operator controlled(Control_ref:Control ref)': '제어모드(0:레시피,1:작업자)',
    'Temperature(T:K)': '발효온도(K)',
    'Heating/cooling water flow rate(Fc:L/h)': '냉난방수유량(L/h)',
    'Heating water flow rate(Fh:L/h)': '가열수유량(L/h)',
    'Generated heat(Q:kJ)': '발생열(kJ)',
    'pH(pH:pH)': 'pH',
    'Acid flow rate(Fa:L/h)': '산투입유량(L/h)',
    'Base flow rate(Fb:L/h)': '염기투입유량(L/h)',
    'Air head pressure(pressure:bar)': '상부압력(bar)',
    'Vessel Volume(V:L)': '발효조부피(L)',
    'Vessel Weight(Wt:Kg)': '발효조중량(kg)',
    'Agitator RPM(RPM:RPM)': '교반속도(RPM)',
    'Aeration rate(Fg:L/h)': '공기주입유량(L/h)',
    'Dissolved oxygen concentration(DO2:mg/L)': '용존산소(mg/L)',
    'Oxygen Uptake Rate(OUR:(g min^{-1}))': '산소소모율(g/min)',
    'Oxygen in percent in off-gas(O2:O2  (%))': '배가스산소(%)',
    'carbon dioxide percent in off-gas(CO2outgas:%)': '배가스이산화탄소(%)',
    'Carbon evolution rate(CER:g/h)': '이산화탄소발생률(g/h)',
    'Substrate concentration(S:g/L)': '기질농도(g/L)',
    'Sugar feed rate(Fs:L/h)': '당공급유량(L/h)',
    'Water for injection/dilution(Fw:L/h)': '희석수유량(L/h)',
    'Oil flow(Foil:L/hr)': '오일유량(L/h)',
    'PAA flow(Fpaa:PAA flow (L/h))': 'PAA투입유량(L/h)',
    'Ammonia shots(NH3_shots:kgs)': '암모니아투입량(kg)',
    'Dumped broth flow(Fremoved:L/h)': '배양액배출유량(L/h)',
    'Penicillin concentration(P:g/L)': '페니실린농도(g/L)',
    'Offline Penicillin concentration(P_offline:P(g L^{-1}))': '페니실린농도_오프라인(g/L)',
    'Offline Biomass concentratio(X_offline:X(g L^{-1}))': '바이오매스농도_오프라인(g/L)',
    'Viscosity(Viscosity_offline:centPoise)': '점도_오프라인(cP)',
    'PAA concentration offline(PAA_offline:PAA (g L^{-1}))': 'PAA농도_오프라인(g/L)',
    'NH_3 concentration off-line(NH3_offline:NH3 (g L^{-1}))': '암모니아농도_오프라인(g/L)',
    'Penicllin_harvested_during_batch(kg)': '중간수확량(kg)',
    'Penicllin_harvested_end_of_batch (kg)': '종료수확량(kg)',
    'Penicllin_yield_total (kg)': '총수확량(kg)',
    'Fault reference(Fault_ref:Fault ref)': '구간결함(0:정상,1:결함)',
    'Fault ref(0-NoFault 1-Fault)': '배치결함(0:정상,1:결함)',
}

assert len(COLUMN_MAP) == 39, '컬럼 매핑은 원본 39개를 모두 포함해야 합니다.'
assert set(merged.columns) == set(COLUMN_MAP), '원본 스키마가 매핑 정의와 다릅니다.'
assert len(set(COLUMN_MAP.values())) == 39, '변경 후 컬럼명이 중복됩니다.'

### 2. 유사한 feature끼리 정렬

feature는 제어 정보, 열·pH, 설비, 산소·배가스, 원료·첨가, 생산 상태 순으로 묶는다. 별도 구분용 컬럼을 추가하지 않고 순서만으로 그룹을 표현한다.

In [ ]:
COLUMN_GROUPS = {
    '번호': [
        '배치번호', '배치참조번호',
    ],
    '시간': [
        '발효시간(h)',
    ],
    'feature_제어': [
        '제어모드(0:레시피,1:작업자)',
    ],
    'feature_열·pH': [
        '발효온도(K)', '냉난방수유량(L/h)', '가열수유량(L/h)', '발생열(kJ)',
        'pH', '산투입유량(L/h)', '염기투입유량(L/h)',
    ],
    'feature_설비': [
        '상부압력(bar)', '발효조부피(L)', '발효조중량(kg)', '교반속도(RPM)',
    ],
    'feature_산소·배가스': [
        '공기주입유량(L/h)', '용존산소(mg/L)', '산소소모율(g/min)',
        '배가스산소(%)', '배가스이산화탄소(%)', '이산화탄소발생률(g/h)',
    ],
    'feature_원료·첨가': [
        '기질농도(g/L)', '당공급유량(L/h)', '희석수유량(L/h)', '오일유량(L/h)',
        'PAA투입유량(L/h)', '암모니아투입량(kg)',
    ],
    'feature_생산': [
        '배양액배출유량(L/h)', '페니실린농도(g/L)',
    ],
    '오프라인': [
        '페니실린농도_오프라인(g/L)', '바이오매스농도_오프라인(g/L)',
        '점도_오프라인(cP)', 'PAA농도_오프라인(g/L)', '암모니아농도_오프라인(g/L)',
    ],
    '결과': [
        '중간수확량(kg)', '종료수확량(kg)', '총수확량(kg)',
        '구간결함(0:정상,1:결함)', '배치결함(0:정상,1:결함)',
    ],
}

COLUMN_ORDER = [column for columns in COLUMN_GROUPS.values() for column in columns]
assert len(COLUMN_ORDER) == 39
assert len(set(COLUMN_ORDER)) == 39
assert set(COLUMN_ORDER) == set(COLUMN_MAP.values())

### 3. 이름 변경 및 저장

In [ ]:
merged_ko = merged.rename(columns=COLUMN_MAP).loc[:, COLUMN_ORDER]

# 컬럼명과 순서만 바뀌었는지 검증한다.
assert merged_ko.shape == merged.shape
for original_name, korean_name in COLUMN_MAP.items():
    assert merged[original_name].equals(merged_ko[korean_name])

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
DICTIONARY_PATH.parent.mkdir(parents=True, exist_ok=True)
merged_ko.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')

group_by_column = {column: group for group, columns in COLUMN_GROUPS.items() for column in columns}
column_dictionary = pd.DataFrame({
    '순서': range(1, len(COLUMN_ORDER) + 1),
    '구분': [group_by_column[column] for column in COLUMN_ORDER],
    '한글컬럼명': COLUMN_ORDER,
    '원본컬럼명': [{new: old for old, new in COLUMN_MAP.items()}[column] for column in COLUMN_ORDER],
})
column_dictionary.to_csv(DICTIONARY_PATH, index=False, encoding='utf-8-sig')

## Checks

그룹별 컬럼 범위와 변환 결과를 확인한다.

In [ ]:
start = 1
group_ranges = []
for group, columns in COLUMN_GROUPS.items():
    end = start + len(columns) - 1
    group_ranges.append({'구분': group, '컬럼범위': f'{start}~{end}', '컬럼수': len(columns)})
    start = end + 1

saved = pd.read_csv(OUTPUT_PATH, nrows=5)
assert saved.columns.tolist() == COLUMN_ORDER

print(f'저장 완료: {OUTPUT_PATH}')
print(f'컬럼 정의: {DICTIONARY_PATH}')
print(f'결과 크기: {merged_ko.shape[0]:,}행 × {merged_ko.shape[1]}열')
display(pd.DataFrame(group_ranges))
display(column_dictionary)
display(merged_ko.head())

## Next Steps

이후 전처리와 EDA에서는 `data/interim/merged_data_ko.csv`를 사용한다. `배치참조번호`는 `배치번호`와 동일하므로, 모델링 직전 중복 식별자 제거 여부를 별도로 결정한다.